# P116 — Por qué Johnny no sabe hacer prompts: cómo los no expertos intentan (y fallan) diseñar prompts

## 1. Título y paper

**Paper:** *Why Johnny Can't Prompt: How Non-AI Experts Try (and Fail) to Design LLM Prompts*  
**Autoría:** J.D. Zamfirescu-Pereira, Richmond Y. Wong, Bjoern Hartmann, Qian Yang  
**Año y venue:** 2023 · CHI '23  
**Nivel:** L2 · **Motor:** `gestion_de_prompts`  
**Ficha completa:** [`P116_gestion_de_prompts`](../../papers/foundational/P116_gestion_de_prompts/README.md)

**Hito:** Documenta con usuarios reales que iterar prompts sin conjunto de evaluación produce mejoras imaginarias, y por qué la intuición falla sistemáticamente.

- [doi:10.1145/3544548.3581388](https://doi.org/10.1145/3544548.3581388)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Escribir prompts parece accesible a cualquiera, y por eso se hace sin ninguna disciplina de ingeniería: sin versionar, sin conjunto de evaluación y mirando dos o tres ejemplos. Con muestras pequeñas, el ruido tiene el mismo tamaño que las mejoras que se buscan.
2. Ejecutar una implementación mínima de la propuesta: Un estudio con participantes no expertos que documenta sus estrategias reales, identifica el patrón dominante —iteración oportunista basada en anécdotas— y argumenta que el prompt necesita las prácticas del software: versionado, evaluación fija y una hipótesis por cambio.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P62
- P63


## 4. Intuición

Escribir prompts parece accesible a cualquiera, y por eso se hace sin ninguna disciplina: se cambia algo, se miran dos ejemplos y se conserva lo que «va mejor». Con muestras pequeñas, el ruido tiene exactamente el mismo tamaño que las mejoras que se buscan.


## 5. Concepto mínimo

```text
Con n ejemplos y calidad p, la desviación de la medida es √(p(1−p)/n)

    n = 20   →  ±0,10      ← comparable a la diferencia entre versiones
    n = 200  →  ±0,03

Iterar sobre 20 ejemplos es elegir ruido.
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('gestion_de_prompts', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. Tres versiones tienen la misma calidad real y una es mejor. ¿Cuál elige quien mira 20 ejemplos?
2. ¿Cuánto ruido tiene esa medida?
3. ¿Qué distingue la iteración sistemática de la oportunista?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('gestion_de_prompts', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('gestion_de_prompts', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

Midiendo sobre 20 ejemplos se elige **v1**, y la mejor de verdad es **v3**: no acierta. La desviación esperada con 20 ejemplos es **0,102**, comparable a la diferencia real entre versiones; con 200 baja a **0,032**.


## 10. Comentario pedagógico

El artículo es un estudio con personas, no un experimento numérico, y su hallazgo es conductual: los no expertos iteran de forma **oportunista** —cambian, miran un par de ejemplos, se quedan con lo que parece mejor— y confunden ruido con mejora de forma sistemática. La alternativa no es más talento redactando: es conjunto de evaluación fijo, una hipótesis por cambio y versionar el prompt con el código.


## 11. Error o anti-patrón deliberado

Anti-patrón: iterar el prompt mirando ejemplos sueltos.


In [ ]:
print('«Con este cambio responde mejor» tras mirar tres ejemplos no es una medida.')
print('Es la misma falacia que reportar la mejor semilla: se elige ruido con confianza.')
print('Y como el prompt no esta versionado, ni siquiera se puede volver atras.')

## 12. Corrección

La disciplina mínima:


In [ ]:
r = run_paper_lab('gestion_de_prompts', seed=7)['result']
for v in r['versiones']:
    print(f"  {v['version']}  real={v['calidad_real']}  en 20={v['medido_en_20_ejemplos']}"
          f"  en 200={v['medido_en_200_ejemplos']}")
print()
print('elegida con 20 ejemplos:', r['elegida_mirando_20_ejemplos'], '| mejor real:', r['mejor_real'])
print('practica sistematica   :', r['practicas']['sistematica'])

## 13. Desafío guiado

Calcula cuántos ejemplos harían falta para distinguir con confianza una mejora de 0,05.


In [ ]:
r = run_paper_lab('gestion_de_prompts', seed=3)['result']
show(r)

## 14. Desafío autónomo

Coge un prompt de tu trabajo, construye un conjunto de evaluación de al menos 100 casos con respuesta esperada, y vuelve a medir las versiones que ya habías descartado.


## 15. Evidencia de aprendizaje

Guarda tu conjunto de evaluación y la comparación de versiones medidas sobre él.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P116_gestion_de_prompts/README.md) · evaluación formal: [`assessments/papers/P116_gestion_de_prompts.md`](../../assessments/papers/P116_gestion_de_prompts.md)


## 16. Cierre

Los prompts ya se gestionan como código. Falta lo mismo un nivel más arriba: cómo se opera un agente que da muchos pasos.


## 17. Conexión con el siguiente hito

- P117

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
